# The Auction Algorithm for the Assignment Problem [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xavimaass/computational_optimal_transport_2026/blob/main/assignments/HW2.ipynb)

**Student Name:** [Your Name Here]

The goal of this assignment is to implement the **Auction Algorithm** seen in the lectures.

Recall the optimal assignment problem on $N$ points:
$$\min_{\sigma \in \mathfrak{S}_N} \sum_{i=1}^N C_{i, \sigma(i)}$$
where $\mathfrak{S}_N$ is the set of all permutations of $\{1, \dots , N\}$. The corresponding dual problem (which is what we'll work with) is:
$$\max_{\phi, \psi \in \mathbb{R}^N} \sum_{i=1}^N \phi_{i} + \sum_{j=1}^N\psi_j \;\; \text{s.t.}\;\; \phi_i + \psi_j \leq c_{ij},\; \forall i,j.$$

We can interpret the problem as having $N$ buyers competing to buy $N$ objects, where:
- $c_{ij}$ is the transport cost of object $j$ to buyer $i$.
- $-\psi_j$ is the (dynamic) price of object $j$.
- $\phi_i$ is what buyer $i$ pays.


## The Auction Algorithm

Just as a reminder, we recall what we saw in the lectures regarding the implementation of the Auction Algorithm.

**Initialization:** Set $\psi_j^{(0)} = 0$ for all $j \in \{1, \ldots, N\}$. Initialize a partial list of assignments $A^{(k)}$

At each iteration $k = 0, 1, 2, \ldots$:
1. If all buyers are assigned, **terminate**. If not, select an unassigned buyer $i \notin A^{(k)}$.
2. Compute the best object for buyer $i$: $$j^* = \arg\min_{j} \bigl(c_{ij} - \psi_j^{(k)}\bigr)$$
In this round, buyer $i$ makes a bid for object $j^*$: so assign $j^*$ to $i$, and delete $j^*$'s previous assignment in $A^{(k)}$, if there was any.
3. Buyer $i$ increases $j^*$'s price maximally while ensuring $j^*$ remains their best choice, and adds $\varepsilon$: $$\Delta_{\varepsilon} = \min_{j \neq j^*}\bigl(c_{ij} - \psi_j^{(k)}\bigr) - \bigl(c_{ij^*} - \psi_{j^*}^{(k)}\bigr) + \varepsilon \geq \varepsilon > 0 $$
$$\psi_{j^*}^{(k+1)} = \psi_{j^*}^{(k)} - \Delta_{\varepsilon},\qquad (\text{optionally:}\;\;\phi_{i}^{(k+1)} = c_{ij^{*}} - \psi_{j^*}^{(k)} - \varepsilon)$$

**Exercise 1:** **Implement the Auction Algorithm:** Complete the `auction_algorithm` function with your own code. The function should take a cost matrix `C` and the perturbation parameter `epsilon` as input.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Set a seed for reproducibility of random results
np.random.seed(42)

In [ ]:
def auction_algorithm(C, epsilon, prices=None):
    """
    Standard auction algorithm for the assignment problem (minimization).

    Args:
        C (np.ndarray): Square cost matrix.
        epsilon (float): Perturbation parameter.
        prices (np.ndarray | None): Optional warm-start prices to initialize the algorithm (typically np.zeros).

    Returns:
        tuple: (assignment, prices, bid_count) where 
            assignment is a dictionary mapping {person: object}, 
            prices is the final price vector,
            bid_count is counts the total number of bids placed.
    """
    n = C.shape[0]
    if n != C.shape[1]:
        raise ValueError("Cost matrix must be square.")
    if epsilon <= 0:
        raise ValueError("epsilon must be strictly positive.")

    if prices is None:
        prices = np.zeros(n, dtype=float)
    else:
        prices = np.array(prices, dtype=float, copy=True)

    object_to_person = {}
    bid_count = 0

    # YOUR CODE HERE
    # FEEL FREE TO MODIFY THE CODE AS YOU DESIRE; AS LONG AS YOU MAINTAIN THE FUNCTION SIGNATURE.

    assignment = {person: obj for obj, person in object_to_person.items()}
    return assignment, prices, bid_count

**Exercise 2:** **Sanity Check.** Consider the following simple example, with $N=3$ and the cost matrix presented below.

Recall that in Practical Assignment 1, we learned how to solve an OT problem using a linear solver, such as `cvxpy`. Implement a solution to this simple OT problem using this linear programming approach.

Compare the solution to the one obtained by applying the auction algorithm you just implemented.

_Hint: In this very simple example, solutions should coincide. You should obtain an optimal assignment $0\to1, 1\to0, 2\to2$ with a total cost of 6_

In [ ]:
n_simple = 3
C_simple = np.array([
    [10, 1, 5],   # Costs for Person 0 for 0, 1, 2
    [2, 7, 8],    # Costs for Person 1
    [9, 4, 3]     # Costs for Person 2
])
# Person 0's best choice is 1 (cost 1). Person 1's best choice is 0 (cost 2). Person 2's best choice is 2 (cost 3).
# The optimal assignment is thus (by inspection): 0->1, 1->0, 2->2
# Total optimal cost = C[0,1] + C[1,0] + C[2,2] = 1 + 2 + 3 = 6

In [ ]:
import cvxpy as cp

# YOUR CODE HERE: Formulate and solve the linear programming relaxation of the assignment problem using cvxpy.

# Extract and print the results
lp_cost = None # the problem.value
lp_assignment = None # Hint: {i: np.argmax(lp_assignment_matrix[i, :]) for i in range(n_simple)}

print(f"Optimal Assignment from LP: {lp_assignment}")
print(f"Total Cost from LP: {lp_cost}\n")

In [ ]:
def assignment_cost(C, assignment):
    return float(sum(C[i, assignment[i]] for i in range(C.shape[0])))

In [ ]:
# Use epsilon = 1/(n+1) as proposed in the lectures.
epsilon_simple = 1 / (n_simple + 1)

auction_assignment, _, auction_bid_count = auction_algorithm(C_simple, epsilon=epsilon_simple)
auction_cost = assignment_cost(C_simple, auction_assignment)

print(f"Optimal Assignment from Auction: {auction_assignment}")
print(f"Total Cost from Auction: {auction_cost}")
print(f"Number of bids: {auction_bid_count}\n")

In [ ]:
print("--- Comparison ---")
if auction_cost == lp_cost and auction_assignment == lp_assignment:
    print("Success! The auction algorithm's result matches the linear program's optimal solution.")
else:
    print("Warning: The results do not match. Please check the implementation.")


**Exercise 3:**  **Testing the theory.** 

**a)** Run the following experiment:
- For `N` ranging from 10 to 100, you will generate random cost matrices `C` of size `N x N`. The entries `c_ij` should be random integers between 1 and 10.
- Set `epsilon = 1 / (N + 1)`.
- For each value of `N`, run your auction algorithm 20 times with a new random cost matrix each time.
- At the end, calculate the average number of bids for each `N` over the different repetitions.

For this purpose, complete the `run_experiment` function provided below.

In [ ]:
def run_experiment(N_values, num_repetitions, max_cost):
    """Runs the auction algorithm experiment for different values of N and returns the average number of bids for each N."""
    average_bids = []

    # Main loop for the experiment
    for N in tqdm(N_values, desc="Running experiments for different N"):
        bids_for_N = []
        for _ in range(num_repetitions):
            # YOUR CODE HERE
            pass
        pass
        # Calculate and store the average number of bids
    return average_bids

In [ ]:
# Experiment Parameters
N_values = range(10, 101, 1)  # N from 10 to 100 in steps of 1
num_repetitions = 20
max_cost = 10

average_bids = run_experiment(N_values, num_repetitions, max_cost)

**b)** Having the results from the previous part:
- Use the following function `plot_results` to create a simple plot showing the average number of bids versus `N`. 
- The plot includes an heuristic fit of the curve `c N**2`, which should represent the theoretical time-complexity of the algorithm.
- Compare the trend you observe in your plot to the theoretical complexity of the algorithm, which is `O(N^2 * C)`. Does your empirical result align with the theory? Explain why or why not.

In [ ]:
def plot_results(N_values, average_bids):
    x = np.array(N_values, dtype=float)
    y = np.array(average_bids, dtype=float)
    fit_first = 20
    c_theory = np.dot(y[:fit_first], x[:fit_first]**2) / np.dot(x[:fit_first]**2, x[:fit_first]**2)  # fit y ≈ c N^2

    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'o-', label='Empirical Average Bids')
    plt.plot(x, c_theory * x**2, 'r--', label=fr'Fitted $cN^2$ (c={c_theory:.3g})')

    plt.title('Average Number of Bids vs. Problem Size (N)')
    plt.xlabel('N (Number of People/Objects)')
    plt.ylabel('Average Number of Bids (over repetitions)')
    plt.ylim(0, 1.1*max(y))
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_results(N_values, average_bids)

[YOUR ANSWER HERE]

**Exercise 4:** **$\varepsilon$-scaling Auction Algorithm.**

Implement the ε-scaling variant of the auction algorithm. This strategy reduces the total number of bids by starting with a large epsilon (coarse scale) and gradually decreasing it, using warm-start prices from each phase to initialize the next (this is why we included the "prices" parameter in our original implementation!).

**Implementation Steps:**

Implement the function `auction_algorithm_epsilon_scaling(C, epsilon_min=None, alpha=5.0)` as follows:

**Initialization:** Set `epsilon_min = 1/(n+1)` if not provided (this guarantees optimality).
Compute `epsilon_0 = max(1.0, max(C) - min(C))` as the initial (coarsest) epsilon.
Initialize `prices = np.zeros(n)` and `total_bid_count = 0`.

**Scaling loop:** While `epsilon > epsilon_min`:

- Run `auction_algorithm(C, epsilon=epsilon, prices=prices)`.
- Accumulate the bid count and record epsilon in a list for diagnostics.
- Reduce: `epsilon = epsilon/alpha`.


**Final phase:** Run `auction_algorithm` once more, but now with `epsilon = epsilon_min` and the warm-start prices from the last scaling phase.

In [ ]:
def auction_algorithm_epsilon_scaling(C, epsilon_min=None, alpha=5.0):
    """
    Epsilon-scaling auction algorithm for a minimization assignment problem.

    Returns:
        assignment (dict): person -> object
        total_bid_count (int): total bids across all scales
        final_prices (np.ndarray): final dual prices
        epsilons_used (list): epsilon values used at each phase
    """
    n = C.shape[0]
    if n != C.shape[1]:
        raise ValueError("Cost matrix must be square.")
    if alpha <= 1:
        raise ValueError("alpha must be > 1.")

    if epsilon_min is None:
        epsilon_min = 1.0 / (n + 1)

    cost_spread = float(np.max(C) - np.min(C))
    epsilon_0 = max(1.0, cost_spread)

    prices = np.zeros(n, dtype=float)
    total_bid_count = 0
    epsilons_used = []

    epsilon = epsilon_0
    while epsilon > epsilon_min:
        # YOUR CODE HERE
        # by the end of the loop you should have the updated `prices`
        pass

    # YOUR CODE HERE: Last run with epsilon_min.
    assignment, prices = None

    return assignment, prices, total_bid_count, epsilons_used

**Exercise 5:** **a) Sanity check.** 

Test the epsilon-scaling algorithm against your vanilla auction implementation on a moderate-sized random instance to verify correctness.
For this, generate a random cost matrix `C` of size `n_test = 300` with integer entries in $\{1, …, 100\}$.
Run both algorithms with the same `epsilon = 1/(n_test + 1)` (`epsilon_min` for the scaling variant).

Verify that the final assignment cost is the same, and compare the total number of bids. What do you observe?

In [ ]:
# --- Quick comparison on one random instance ---
n_test = 300
C_test = None

# YOUR CODE HERE:

# By the end, you should print:
#print(f"Reference auction (single epsilon={epsilon_ref:.4f}): bids={bids_ref}, cost={cost_ref:.0f}")
#print(f"Scaled auction ({len(epsilons_used)} phases): bids={bids_scaled}, cost={cost_scaled:.0f}")

[YOUR ANSWER HERE]

**b) Scaling with $C_{max}:=\|C\|_{\infty}=\max_{i,j} C_{i,j}$** 
Recall that the vanilla auction requires $O(N^2 · C_{max})$ bids, while $\varepsilon$-scaling requires roughly $O(N^2 · \log(C_{max}))$ bids. As $C_max$ grows, the gap between the two should widen.

Run a controlled experiment to study how the scaling advantage grows with the magnitude of the cost entries. For this, complete the `run_scaling_experiment` function below. 
Then, study what happens for Fix `n = 300` and `max_cost_values = 10^(linspace(1, 6, 10))` with `num_repetitions=20`
Use the provided `plot_comparison` to obtain a relevant visualization, and comment on the results you observe.

In [ ]:
def run_scaling_experiment(max_cost_values, n, num_repetitions=10, alpha=5.0):
    """Compare vanilla and epsilon-scaling auction across different cost ranges."""
    bids_vanilla = []
    bids_scaled = []

    for max_cost in tqdm(max_cost_values, desc="Scaling experiment (max_cost)"):
        vanilla_trials = []
        scaled_trials = []

        for _ in range(num_repetitions):
            # YOUR CODE HERE.
            pass
        pass

    return max_cost_values, bids_vanilla, bids_scaled

In [ ]:
max_cost_values = None # YOUR CODE HERE
max_cost_values, bids_vanilla, bids_scaled = run_scaling_experiment(max_cost_values=max_cost_values, n=300, num_repetitions=20, alpha=5.0)

In [ ]:
def plot_comparison(max_cost_values, bids_vanilla, bids_scaled):
    plt.figure(figsize=(7,5))
    plt.plot(max_cost_values, bids_vanilla, marker='o', label='Vanilla auction')
    plt.plot(max_cost_values, bids_scaled, marker='s', label='Epsilon-scaling')

    plt.xscale('log')

    plt.xlabel('max cost (||C||_∞)')
    plt.ylabel('number of bids')
    plt.title('Scaling of Auction Algorithms')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_comparison(max_cost_values, bids_vanilla, bids_scaled)

[YOUR ANSWER HERE]